In [7]:
import sys
!{sys.executable} -m pip install gensim==4.4.0 nltk

  Using cached nltk-3.10.4-py3-none-any.whl.metadata (5.0 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached regex-2026.9.29-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached nltk-3.10.4-py3-none-any.whl (1.7 MB)
Using cached regex-2026.9.29-cp313-cp313-win_amd64.whl (280 kB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)

   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ----- ---------------------------------- 1/

In [8]:
import nltk
import re
import numpy as np

from nltk.corpus import gutenberg, stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Word2Vec

nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\rutuj\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\gutenberg.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rutuj\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rutuj\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rutuj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rutuj\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [9]:
text = gutenberg.raw('carroll-alice.txt')

print(text[:500])

[Alice's Adventures in Wonderland by Lewis Carroll 1865]

CHAPTER I. Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it had no pictures or conversations in
it, 'and what is the use of a book,' thought Alice 'without pictures or
conversation?'

So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy an


In [10]:
sentences = sent_tokenize(text)

stop_words = set(stopwords.words('english'))

data = []

for sentence in sentences:
    
    sentence = sentence.lower()
    
    # Remove punctuation, special characters and digits
    sentence = re.sub(r'[^a-z\s]', '', sentence)
    
    words = word_tokenize(sentence)
    
    # Remove stop words
    words = [word for word in words if word not in stop_words]
    
    if len(words) > 1:
        data.append(words)

print(data[:3])

[['alices', 'adventures', 'wonderland', 'lewis', 'carroll', 'chapter'], ['rabbithole', 'alice', 'beginning', 'get', 'tired', 'sitting', 'sister', 'bank', 'nothing', 'twice', 'peeped', 'book', 'sister', 'reading', 'pictures', 'conversations', 'use', 'book', 'thought', 'alice', 'without', 'pictures', 'conversation'], ['considering', 'mind', 'well', 'could', 'hot', 'day', 'made', 'feel', 'sleepy', 'stupid', 'whether', 'pleasure', 'making', 'daisychain', 'would', 'worth', 'trouble', 'getting', 'picking', 'daisies', 'suddenly', 'white', 'rabbit', 'pink', 'eyes', 'ran', 'close']]


In [11]:
model = Word2Vec(
    data,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,
    epochs=10
)

print("CBOW model trained successfully!")

CBOW model trained successfully!


In [12]:
print("Vocabulary size:", len(model.wv))
print(model.wv.index_to_key[:20])

Vocabulary size: 1331
['said', 'alice', 'little', 'one', 'know', 'like', 'would', 'went', 'could', 'thought', 'time', 'queen', 'see', 'king', 'dont', 'began', 'im', 'well', 'turtle', 'mock']


In [13]:
word = "alice"

print("Words similar to", word)

for w, score in model.wv.most_similar(word, topn=5):
    print(w, score)

Words similar to alice
quite 0.9997335076332092
round 0.999729335308075
little 0.9997169971466064
one 0.9997091293334961
said 0.9997031092643738


In [14]:
context = ["white", "rabbit"]

vectors = []

for word in context:
    if word in model.wv:
        vectors.append(model.wv[word])

context_vector = np.mean(vectors, axis=0)

predictions = model.wv.similar_by_vector(context_vector, topn=5)

print("Context:", context)
print("Predicted words:")

for word, score in predictions:
    print(word, score)

Context: ['white', 'rabbit']
Predicted words:
white 0.9998327493667603
rabbit 0.9998120069503784
little 0.9997577667236328
one 0.999739408493042
time 0.9996981024742126


In [15]:
print("Similarity between alice and rabbit:",
      model.wv.similarity("alice", "rabbit"))

print("Similarity between alice and queen:",
      model.wv.similarity("alice", "queen"))

Similarity between alice and rabbit: 0.9993768
Similarity between alice and queen: 0.9996502


In [16]:
print("CBOW Model Evaluation")
print("---------------------")
print("Vocabulary size:", len(model.wv))
print("Vector size:", model.vector_size)
print("Window size:", model.window)
print("Model: CBOW")

CBOW Model Evaluation
---------------------
Vocabulary size: 1331
Vector size: 100
Window size: 5
Model: CBOW
